Fechas:

Fecha defuncion 
Fecha sintomas 
Fecha ingreso 


Fecha ingreso - Fecha sintomas = DIAS_SINTOMAS


Fecha defuncion - Fecha de Ingreso  = DIAS_ATENCION

Fecha defuncion - Fecha de sitoma = DIAS_ENFERMO

In [2]:
from src.database import get_database

db = get_database()

In [5]:
for key in db.mexico_city_covid_data.find_one().keys():
    print(key, type(key))


_id <class 'str'>
ASMA <class 'str'>
CARDIOVASCULAR <class 'str'>
CLASIFICACION_FINAL_COVID <class 'str'>
CLASIFICACION_FINAL_FLU <class 'str'>
DIABETES <class 'str'>
EDAD <class 'str'>
EMBARAZO <class 'str'>
ENTIDAD_NAC <class 'str'>
ENTIDAD_RES <class 'str'>
ENTIDAD_UM <class 'str'>
EPOC <class 'str'>
FECHA_DEF <class 'str'>
FECHA_INGRESO <class 'str'>
FECHA_SINTOMAS <class 'str'>
HABLA_LENGUA_INDIG <class 'str'>
HIPERTENSION <class 'str'>
INDIGENA <class 'str'>
INMUSUPR <class 'str'>
INTUBADO <class 'str'>
MIGRANTE <class 'str'>
MUNICIPIO_RES <class 'str'>
NACIONALIDAD <class 'str'>
NEUMONIA <class 'str'>
OBESIDAD <class 'str'>
ORIGEN <class 'str'>
OTRA_COM <class 'str'>
OTRO_CASO <class 'str'>
PAIS_NACIONALIDAD <class 'str'>
PAIS_ORIGEN <class 'str'>
RENAL_CRONICA <class 'str'>
RESULTADO_ANTIGENO <class 'str'>
RESULTADO_PCR <class 'str'>
RESULTADO_PCR_COINFECCION <class 'str'>
SECTOR <class 'str'>
SEXO <class 'str'>
TABAQUISMO <class 'str'>
TIPO_PACIENTE <class 'str'>
TOMA_MUESTRA_

In [7]:
print(db.mexico_city_covid_data.find_one({}, {"FECHA_DEF": 1, "FECHA_INGRESO": 1, "FECHA_SINTOMAS": 1, "_id": 0}))


{'FECHA_DEF': '9999-99-99', 'FECHA_INGRESO': '2020-08-18', 'FECHA_SINTOMAS': '2020-08-17'}


In [3]:
import polars as pl
import pyarrow.parquet as pq
from pathlib import Path

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data" / "polars"
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)


cursor = db.mexico_city_covid_data.find({}, batch_size=100_000)

writer = None
batch = []

for doc in cursor:
    
    batch.append(doc)

    if len(batch) >= 100_000:

        df = pl.DataFrame(batch)

        table = df.to_arrow()

        if writer is None:
            writer = pq.ParquetWriter(
                DATA_DIR / "mexico_city_covid_data_complete.parquet",
                table.schema
            )

        writer.write_table(table)

        batch = []

if batch:
    df = pl.DataFrame(batch)
    table = df.to_arrow()

    if writer is None:
        writer = pq.ParquetWriter(
            DATA_DIR / "mexico_city_covid_data_complete.parquet",
            table.schema
        )

    writer.write_table(table)

if writer is not None:
    writer.close()

In [4]:
from pathlib import Path
import polars as pl

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data" / "polars"

input_file = DATA_DIR / "mexico_city_covid_data_complete.parquet"
temp_file = DATA_DIR / "mexico_city_covid_data_complete_temp.parquet"

df_lazy = pl.scan_parquet(input_file)
date_columns = ["FECHA_DEF", "FECHA_INGRESO", "FECHA_SINTOMAS"]

processed_df = df_lazy.with_columns([
    pl.col(col)
    .replace("9999-99-99", None)
    .str.to_date("%Y-%m-%d", strict=False)
    for col in date_columns
])

processed_df.sink_parquet(temp_file)

input_file.unlink()        
temp_file.rename(input_file)

print("File updated successfully")


File updated successfully


Fecha ingreso - Fecha sintomas = DIAS_SINTOMAS


Fecha defuncion - Fecha de Ingreso  = DIAS_ATENCION

Fecha defuncion - Fecha de sitoma = DIAS_ENFERMO

In [5]:
from pathlib import Path
import polars as pl

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data" / "polars"

input_file = DATA_DIR / "mexico_city_covid_data_complete.parquet"
output_file = DATA_DIR / "mexico_city_covid_data_dates.parquet"


df_lazy = pl.scan_parquet(input_file)

# Creamos la nueva tabla con la primera lógica
processed_df = df_lazy.with_columns([
    # Fecha ingreso - Fecha sintomas = DIAS_SINTOMAS
    (pl.col("FECHA_INGRESO") - pl.col("FECHA_SINTOMAS"))
    .dt.total_days()
    .alias("DIAS_SINTOMAS"),
    # Fecha defuncion - Fecha de Ingreso  = DIAS_ATENCION
    (pl.col("FECHA_DEF") - pl.col("FECHA_INGRESO"))
    .dt.total_days()
    .alias("DIAS_ATENCION"),
    # Fecha defuncion - Fecha de sitoma = DIAS_ENFERMO
    (pl.col("FECHA_DEF") - pl.col("FECHA_SINTOMAS"))
    .dt.total_days()
    .alias("DIAS_ENFERMO")
])

processed_df.sink_parquet(output_file)

print(f"Nuevo archivo creado exitosamente en: {output_file}")

Nuevo archivo creado exitosamente en: /home/netfoor/Documents/university/KDD/covid19/data/polars/mexico_city_covid_data_dates.parquet


In [6]:
schema = pl.read_parquet_schema(output_file)
print(schema.keys())

odict_keys(['_id', 'ASMA', 'CARDIOVASCULAR', 'CLASIFICACION_FINAL_COVID', 'CLASIFICACION_FINAL_FLU', 'DIABETES', 'EDAD', 'EMBARAZO', 'ENTIDAD_NAC', 'ENTIDAD_RES', 'ENTIDAD_UM', 'EPOC', 'FECHA_DEF', 'FECHA_INGRESO', 'FECHA_SINTOMAS', 'HABLA_LENGUA_INDIG', 'HIPERTENSION', 'INDIGENA', 'INMUSUPR', 'INTUBADO', 'MIGRANTE', 'MUNICIPIO_RES', 'NACIONALIDAD', 'NEUMONIA', 'OBESIDAD', 'ORIGEN', 'OTRA_COM', 'OTRO_CASO', 'PAIS_NACIONALIDAD', 'PAIS_ORIGEN', 'RENAL_CRONICA', 'RESULTADO_ANTIGENO', 'RESULTADO_PCR', 'RESULTADO_PCR_COINFECCION', 'SECTOR', 'SEXO', 'TABAQUISMO', 'TIPO_PACIENTE', 'TOMA_MUESTRA_ANTIGENO', 'TOMA_MUESTRA_LAB', 'UCI', 'YEAR_ORIGIN', 'DIAS_SINTOMAS', 'DIAS_ATENCION', 'DIAS_ENFERMO'])


In [13]:
from pathlib import Path
import polars as pl

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data" / "polars"

input_file = DATA_DIR / "mexico_city_covid_data_dates.parquet"
columns = ["DIAS_SINTOMAS", "DIAS_ATENCION", "DIAS_ENFERMO"]

df_lazy = pl.scan_parquet(input_file)


resultado = df_lazy.select(['DIAS_SINTOMAS', 'DIAS_ATENCION', 'DIAS_ENFERMO']).slice(0, 19).collect()

print(resultado)


shape: (19, 3)
┌───────────────┬───────────────┬──────────────┐
│ DIAS_SINTOMAS ┆ DIAS_ATENCION ┆ DIAS_ENFERMO │
│ ---           ┆ ---           ┆ ---          │
│ i64           ┆ i64           ┆ i64          │
╞═══════════════╪═══════════════╪══════════════╡
│ 1             ┆ null          ┆ null         │
│ 0             ┆ null          ┆ null         │
│ 3             ┆ null          ┆ null         │
│ 5             ┆ null          ┆ null         │
│ 0             ┆ null          ┆ null         │
│ …             ┆ …             ┆ …            │
│ 5             ┆ null          ┆ null         │
│ 0             ┆ null          ┆ null         │
│ 9             ┆ null          ┆ null         │
│ 5             ┆ null          ┆ null         │
│ 1             ┆ null          ┆ null         │
└───────────────┴───────────────┴──────────────┘


In [14]:
resumen = df_lazy.select(['DIAS_SINTOMAS', 'DIAS_ATENCION', 'DIAS_ENFERMO']).collect().describe()

print(resumen)

shape: (9, 4)
┌────────────┬───────────────┬───────────────┬──────────────┐
│ statistic  ┆ DIAS_SINTOMAS ┆ DIAS_ATENCION ┆ DIAS_ENFERMO │
│ ---        ┆ ---           ┆ ---           ┆ ---          │
│ str        ┆ f64           ┆ f64           ┆ f64          │
╞════════════╪═══════════════╪═══════════════╪══════════════╡
│ count      ┆ 7.840211e6    ┆ 94087.0       ┆ 94087.0      │
│ null_count ┆ 0.0           ┆ 7.746124e6    ┆ 7.746124e6   │
│ mean       ┆ 2.40689       ┆ 9.191578      ┆ 14.529659    │
│ std        ┆ 9.13633       ┆ 12.540836     ┆ 13.139388    │
│ min        ┆ -11.0         ┆ -365.0        ┆ -37.0        │
│ 25%        ┆ 0.0           ┆ 2.0           ┆ 7.0          │
│ 50%        ┆ 1.0           ┆ 6.0           ┆ 12.0         │
│ 75%        ┆ 3.0           ┆ 12.0          ┆ 19.0         │
│ max        ┆ 371.0         ┆ 430.0         ┆ 439.0        │
└────────────┴───────────────┴───────────────┴──────────────┘


Defunciones correspondientes a la entidad de residencia 09

In [17]:
from pathlib import Path
import polars as pl

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data" / "polars"

input_file = DATA_DIR / "mexico_city_covid_data_dates.parquet"

df_lazy = (
    pl.scan_parquet(input_file)
    .filter(
        (pl.col("ENTIDAD_RES") == "09") &
        (pl.col("FECHA_DEF").is_not_null())
    )
)

resultado = df_lazy.select(pl.len().alias("Defunciones en CDMX")).collect()
print(resultado)

shape: (1, 1)
┌─────────────────────┐
│ Defunciones en CDMX │
│ ---                 │
│ u32                 │
╞═════════════════════╡
│ 59767               │
└─────────────────────┘


In [19]:
from pathlib import Path
import polars as pl

ROOT = Path.cwd().parent
DATA_DIR = ROOT / "data" / "polars"

input_file = DATA_DIR / "mexico_city_covid_data_dates.parquet"

df_lazy = (
    pl.scan_parquet(input_file)
    .filter(
        (pl.col("ENTIDAD_RES") == "09") &      
        (pl.col("FECHA_DEF").is_not_null())    
    )
    
    .group_by("MUNICIPIO_RES")
    
    .agg(
        pl.len().alias("Defunciones en CDMX")
    )
    
    .sort("Defunciones en CDMX", descending=True)
)

resultado = df_lazy.collect()

print(resultado)

shape: (17, 2)
┌───────────────┬─────────────────────┐
│ MUNICIPIO_RES ┆ Defunciones en CDMX │
│ ---           ┆ ---                 │
│ str           ┆ u32                 │
╞═══════════════╪═════════════════════╡
│ 007           ┆ 11511               │
│ 005           ┆ 9480                │
│ 010           ┆ 4953                │
│ 003           ┆ 4094                │
│ 002           ┆ 4029                │
│ …             ┆ …                   │
│ 011           ┆ 1843                │
│ 008           ┆ 1155                │
│ 004           ┆ 863                 │
│ 009           ┆ 704                 │
│ 999           ┆ 3                   │
└───────────────┴─────────────────────┘


Max: IZTAPALAPA = 11511
Min: MILPA ALTA = 704